# Fault Engine — Train in Colab

Same SensorAutoencoder architecture and training logic as `fault_engine/train.py`, adapted to run from an uploaded CSV instead of a live Postgres connection (Colab can't reach your local database).

**Steps:** run every cell in order. Cell 3 will prompt a file picker — upload `sensor_data_export.csv`. The last cell downloads the trained artifacts (`autoencoder.pt`, `scaler.joblib`, `meta.json`) back to your computer; drop them into `fault_engine/artifacts/` to use them locally.

In [ ]:
!pip install -q torch scikit-learn pandas joblib

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

In [ ]:
from google.colab import files
uploaded = files.upload()  # pick sensor_data_export.csv when prompted
csv_path = list(uploaded.keys())[0]
df = pd.read_csv(csv_path)
print(df.shape)
df.head()

In [ ]:
FEATURE_COLUMNS = [
    "soc_percent", "voltage", "current", "battery_temp", "ambient_temp",
    "charging_duration_min", "degradation_rate", "efficiency", "charging_cycles",
    "battery_capacity_kwh", "energy_consumed_kwh", "charging_duration_hours",
    "charging_rate_kw", "soc_start", "soc_end", "temperature",
]

df = df.dropna(subset=FEATURE_COLUMNS).reset_index(drop=True)

train_df, val_df = train_test_split(df, test_size=0.15, random_state=42)

scaler = StandardScaler()
scaler.fit(train_df[FEATURE_COLUMNS].values)

def scale(frame):
    return scaler.transform(frame[FEATURE_COLUMNS].values).astype(np.float32)

X_train = scale(train_df)
X_val = scale(val_df)
print(f"train={len(X_train)}  val={len(X_val)}")

In [ ]:
class SensorAutoencoder(nn.Module):
    def __init__(self, n_features, hidden_dims=(12, 8), bottleneck_dim=4, dropout=0.1):
        super().__init__()
        encoder_layers = []
        in_dim = n_features
        for h in hidden_dims:
            encoder_layers += [nn.Linear(in_dim, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = h
        encoder_layers.append(nn.Linear(in_dim, bottleneck_dim))
        self.encoder = nn.Sequential(*encoder_layers)

        decoder_layers = []
        in_dim = bottleneck_dim
        for h in reversed(hidden_dims):
            decoder_layers += [nn.Linear(in_dim, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = h
        decoder_layers.append(nn.Linear(in_dim, n_features))
        self.decoder = nn.Sequential(*decoder_layers)

    def forward(self, x):
        return self.decoder(self.encoder(x))

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

EPOCHS = 200
BATCH_SIZE = 256
LR = 1e-3
THRESHOLD_PERCENTILE = 95.0

def reconstruction_errors(model, X, batch_size=1024):
    model.eval()
    errors = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            batch = torch.from_numpy(X[i:i + batch_size]).to(device)
            recon = model(batch)
            err = torch.mean((batch - recon) ** 2, dim=1)
            errors.append(err.cpu().numpy())
    return np.concatenate(errors)

n_features = X_train.shape[1]
train_loader = DataLoader(TensorDataset(torch.from_numpy(X_train)), batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

model = SensorAutoencoder(n_features=n_features).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
criterion = nn.MSELoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)

X_val_t = torch.from_numpy(X_val).to(device)
best_val_loss = float("inf")
best_state = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_losses = []
    for (batch,) in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        recon = model(batch)
        loss = criterion(recon, batch)
        loss.backward()
        optimizer.step()
        epoch_losses.append(loss.item())

    model.eval()
    with torch.no_grad():
        val_loss = criterion(model(X_val_t), X_val_t).item()

    train_loss = float(np.mean(epoch_losses))
    scheduler.step(val_loss)

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{EPOCHS} | train_loss={train_loss:.5f} | val_loss={val_loss:.5f}")

    if val_loss < best_val_loss - 1e-6:
        best_val_loss = val_loss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}

print(f"Completed all {EPOCHS} epochs. Best val_loss={best_val_loss:.5f}")
model.load_state_dict(best_state)
print("Training complete.")

In [ ]:
val_errors = reconstruction_errors(model, X_val)
threshold = float(np.percentile(val_errors, THRESHOLD_PERCENTILE))
print(f"Threshold (P{THRESHOLD_PERCENTILE} of validation error): {threshold:.6f}")

sample = X_val[:300].copy()
injected = sample.copy()
temp_idx = FEATURE_COLUMNS.index("battery_temp")
current_idx = FEATURE_COLUMNS.index("current")
injected[:, temp_idx] += 5.0
injected[:, current_idx] += 5.0
base_err = reconstruction_errors(model, sample)
fault_err = reconstruction_errors(model, injected)
detect_rate = float((fault_err > threshold).mean())
print(f"Mean error on normal samples:         {base_err.mean():.4f}")
print(f"Mean error on injected-fault samples: {fault_err.mean():.4f}")
print(f"Detection rate on injected faults:    {detect_rate*100:.1f}%")

In [ ]:
torch.save(model.state_dict(), "autoencoder.pt")
joblib.dump(scaler, "scaler.joblib")

meta = {
    "feature_columns": FEATURE_COLUMNS,
    "threshold": threshold,
    "threshold_percentile": THRESHOLD_PERCENTILE,
    "n_features": n_features,
    "n_train": len(X_train),
    "n_val": len(X_val),
    "synthetic_fault_detection_rate": detect_rate,
}
with open("meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print("Saved: autoencoder.pt, scaler.joblib, meta.json")

In [ ]:
from google.colab import files
files.download("autoencoder.pt")
files.download("scaler.joblib")
files.download("meta.json")